# 03 · Cache pooled DINOv2 features (TwoRoom)

Runs the frozen DINOv2 ViT-S/14 once over the dataset and stores pooled patch tokens for every variant of
experiment E1, so the predictors in notebook 04 train from disk without touching the encoder again.

| variant | tokens per frame | operation on the 14 x 14 patch grid |
|---|---|---|
| `cls` | 1 | CLS token |
| `mean` | 1 | mean over all patches |
| `g2` | 4 | adaptive average pool to 2 x 2 |
| `g4` | 16 | adaptive average pool to 4 x 4 |
| `g7` | 49 | adaptive average pool to 7 x 7 |

Preprocessing copies the full-grid DINO-WM reference exactly (scale to [-1, 1], resize 224 -> 196 px,
`forward_features` of `dinov2_vits14`); the standalone pipeline was checked against the reference's own
encoder (max difference 5e-4 on token values of order 1). So the token count is the only thing that
differs between the reference and the pooled models.

Features are written as float16 HDF5 files, one per variant, with the episode table and actions copied in.
Extraction is resumable: rerun the extraction cell after a disconnect and it continues where it stopped.

## 1. Paths and environment variables

In [1]:
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = Path("/content")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# must be set before importing stable_worldmodel / mujoco
os.environ["STABLEWM_HOME"] = str(ROOT / "stablewm")      # datasets/ and checkpoints/ live here
os.environ.setdefault("WANDB_MODE", "disabled")
if sys.platform == "linux":
    os.environ.setdefault("MUJOCO_GL", "egl")             # headless rendering; macOS uses its default

HOME = Path(os.environ["STABLEWM_HOME"])
(HOME / "datasets").mkdir(parents=True, exist_ok=True)
print("root:", ROOT)
print("STABLEWM_HOME:", HOME)
print(f"free disk: {shutil.disk_usage(HOME).free / 1e9:.0f} GB")

root: /content
STABLEWM_HOME: /content/stablewm
free disk: 75 GB


## 2. Install (Colab only)

In [2]:
PACKAGES = [
    "stable-worldmodel[train,env]==0.1.1",
    "hdf5plugin",
    "mujoco>=3.14,<3.15",
    "transformers>=4.50,<5",
    "datasets>=3",
    "huggingface-hub<1",
]

if IN_COLAB:
    import subprocess

    def pip(*args):
        cmd = [sys.executable, "-m", "uv", "pip", "install", "--system", *args]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print((result.stdout + result.stderr)[-6000:])      # uv's own error message
            raise RuntimeError("install failed, see the output above")

    # Install system dependency swig required to compile box2d-py
    print("Installing system dependencies (swig)... ")
    subprocess.run(["apt-get", "update", "-y"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "install", "-y", "swig"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

    if sys.version_info >= (3, 13):
        stub = Path("/content/labmaze_stub")
        (stub / "labmaze").mkdir(parents=True, exist_ok=True)
        (stub / "labmaze" / "__init__.py").write_text("# empty stand-in, see notebook section 2\n")
        (stub / "pyproject.toml").write_text(
            "[build-system]\nrequires = [\"setuptools\"]\nbuild-backend = \"setuptools.build_meta\"\n"
            "[project]\nname = \"labmaze\"\nversion = \"1.0.6\"\n"
        )
        pip(str(stub))

    pip(*PACKAGES)
    print("installed on python", sys.version.split()[0])

Installing system dependencies (swig)... 
installed on python 3.13.15


## 3. Imports and settings

In [3]:
import platform
import queue
import threading
import time
import warnings

import h5py
import numpy as np
import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")
import stable_worldmodel as swm

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else DEVICE
SMOKE = DEVICE != "cuda"

VARIANTS = {"cls": 1, "mean": 1, "g2": 4, "g4": 16, "g7": 49}   # name -> tokens per frame
FEAT_DIM = 384                                                  # DINOv2 ViT-S
ENCODER_PX = 196                                                # 14 x 14 patches of 14 px, as in the reference
BATCH = 64 if SMOKE else 256
FEATURES_DIR = HOME / "features"
FEATURES_DIR.mkdir(exist_ok=True)

print("python", platform.python_version(), "| torch", torch.__version__, "| device", GPU)
print("smoke configuration" if SMOKE else "full configuration")

python 3.13.15 | torch 2.11.0+cu130 | device Tesla T4
full configuration


## 4. Dataset

Same download as notebook 00 (12.8 GB `tworoom.h5`, streamed). Without enough disk the locally collected `tworoom_local` is used.

In [4]:
import tarfile
import urllib.request

import zstandard
from tqdm.auto import tqdm

TWOROOM_URL = "https://huggingface.co/datasets/quentinll/lewm-tworooms/resolve/main/tworoom.tar.zst"
MIN_FREE_GB = 20
LOCAL_EPISODES = 20


def download_and_extract(url, dest_dir):
    """Stream a .tar.zst from `url` and extract it into `dest_dir`."""
    with urllib.request.urlopen(url) as resp:
        total = int(resp.headers.get("Content-Length", 0)) or None
        with tqdm.wrapattr(resp, "read", total=total, desc=Path(url).name) as raw:
            with zstandard.ZstdDecompressor().stream_reader(raw) as zr:
                with tarfile.open(fileobj=zr, mode="r|") as tar:
                    tar.extractall(dest_dir)


def collect_local_tworoom(path, episodes):
    from stable_worldmodel.envs.two_room import ExpertPolicy

    world = swm.World("swm/TwoRoom-v1", num_envs=4, image_shape=(224, 224),
                      max_episode_steps=100, render_mode="rgb_array")
    world.set_policy(ExpertPolicy(action_noise=2.0, action_repeat_prob=0.05))
    world.collect(path, episodes=episodes, seed=0, format="hdf5")
    world.close()


free_gb = shutil.disk_usage(HOME).free / 1e9
if (HOME / "datasets" / "tworoom.h5").exists():
    DATASET = "tworoom"
elif free_gb >= MIN_FREE_GB:
    download_and_extract(TWOROOM_URL, HOME / "datasets")
    DATASET = "tworoom"
else:
    DATASET = "tworoom_local"
    if not (HOME / "datasets" / "tworoom_local.h5").exists():
        print(f"only {free_gb:.0f} GB free: collecting {LOCAL_EPISODES} local episodes instead")
        collect_local_tworoom(HOME / "datasets" / "tworoom_local.h5", LOCAL_EPISODES)

dataset = swm.data.HDF5Dataset(DATASET, keys_to_cache=["action", "proprio"])
lengths, offsets = np.asarray(dataset.lengths), np.asarray(dataset.offsets)
print(f"dataset: {DATASET} | {len(lengths)} episodes | {lengths.sum()} steps | columns: {dataset.column_names}")

tworoom.tar.zst:   0%|          | 0/3425937909 [00:00<?, ?it/s]

dataset: tworoom | 10000 episodes | 920809 steps | columns: ['action', 'distance_to_target', 'ep_idx', 'id', 'observation', 'pixels', 'pos_agent', 'pos_target', 'proprio', 'render_time', 'reward', 'step_idx', 'terminated', 'truncated']


In [5]:
H5_PATH = HOME / "datasets" / f"{DATASET}.h5"
with h5py.File(H5_PATH, "r") as f:
    PIX_SHAPE, PIX_DTYPE = f["pixels"].shape, f["pixels"].dtype
    ACTION_DIM = f["action"].shape[1]
    print("pixels", PIX_SHAPE, PIX_DTYPE, "| chunks", f["pixels"].chunks, "| compression", f["pixels"].compression)
    print("action", f["action"].shape, "| columns:", [k for k in f.keys()])
N_FRAMES_TOTAL, N_EPISODES_TOTAL = int(lengths.sum()), len(lengths)

pixels (920809, 224, 224, 3) uint8 | chunks (100, 224, 224, 3) | compression None
action (920809, 2) | columns: ['action', 'distance_to_target', 'ep_idx', 'ep_len', 'ep_offset', 'id', 'observation', 'pixels', 'pos_agent', 'pos_target', 'proprio', 'render_time', 'reward', 'step_idx', 'terminated', 'truncated']


## 5. Encoder, preprocessing and pooling

In [6]:
dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(DEVICE).eval()
dino.requires_grad_(False)


def preprocess(frames_uint8):
    """(B, 224, 224, 3) uint8 numpy -> (B, 3, 196, 196) float on DEVICE, exactly as the reference does it."""
    x = torch.from_numpy(frames_uint8).to(DEVICE).permute(0, 3, 1, 2).float() / 255.0
    x = (x - 0.5) / 0.5
    return F.interpolate(x, size=(ENCODER_PX, ENCODER_PX), mode="bilinear", antialias=True, align_corners=False)


@torch.no_grad()
def encode_and_pool(frames_uint8):
    """Returns {variant: (B, tokens, 384) float16 numpy} for one batch of frames."""
    out = dino.forward_features(preprocess(frames_uint8))
    patches = out["x_norm_patchtokens"]                                   # (B, 196, 384)
    B = patches.shape[0]
    grid = patches.transpose(1, 2).reshape(B, FEAT_DIM, 14, 14)
    if DEVICE == "mps":
        grid = grid.cpu()            # MPS cannot pool 14 -> 4; the tensors are small
    feats = {"cls": out["x_norm_clstoken"][:, None], "mean": patches.mean(1, keepdim=True)}
    for name, k in [("g2", 2), ("g4", 4), ("g7", 7)]:
        feats[name] = F.adaptive_avg_pool2d(grid, k).flatten(2).transpose(1, 2)   # (B, k*k, 384)
    return {name: f.to(torch.float16).cpu().numpy() for name, f in feats.items()}


with h5py.File(H5_PATH, "r") as f:
    sample = encode_and_pool(f["pixels"][:4])
for name, arr in sample.items():
    assert arr.shape == (4, VARIANTS[name], FEAT_DIM), (name, arr.shape)
print({name: arr.shape[1:] for name, arr in sample.items()})

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 376MB/s]


{'cls': (1, 384), 'mean': (1, 384), 'g2': (4, 384), 'g4': (16, 384), 'g7': (49, 384)}


## 6. Timing and size projection

Measures disk read speed and encoder speed separately on a few thousand frames, then projects the time and
storage for the whole dataset and for episode subsets. Read the table before choosing `N_EPISODES` below.

In [7]:
N_TIME = min(N_FRAMES_TOTAL, 512 if SMOKE else 4096)

with h5py.File(H5_PATH, "r") as f:
    t0 = time.perf_counter()
    batches = [f["pixels"][i:i + BATCH] for i in range(0, N_TIME, BATCH)]
    read_s = time.perf_counter() - t0
encode_and_pool(batches[0])                                             # warm-up
if DEVICE == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
for b in batches:
    encode_and_pool(b)
if DEVICE == "cuda":
    torch.cuda.synchronize()
encode_s = time.perf_counter() - t0

read_fps, encode_fps = N_TIME / read_s, N_TIME / encode_s
pipeline_fps = min(read_fps, encode_fps)                                # reading and encoding overlap
print(f"read {read_fps:.0f} frames/s | encode {encode_fps:.0f} frames/s on {GPU} | pipeline about {pipeline_fps:.0f} frames/s\n")

print(f"{'episodes':>9} {'frames':>9} {'time':>8} | " + " ".join(f"{v:>7}" for v in VARIANTS) + "   (GB per variant, float16)")
for n_ep in sorted({1000, 2000, 5000, N_EPISODES_TOTAL}):
    if n_ep > N_EPISODES_TOTAL:
        continue
    n_frames = int(lengths[:n_ep].sum())
    sizes = " ".join(f"{n_frames * k * FEAT_DIM * 2 / 1e9:7.2f}" for k in VARIANTS.values())
    print(f"{n_ep:>9} {n_frames:>9} {n_frames / pipeline_fps / 60:6.0f} min | {sizes}")

read 2855 frames/s | encode 348 frames/s on Tesla T4 | pipeline about 348 frames/s

 episodes    frames     time |     cls    mean      g2      g4      g7   (GB per variant, float16)
     1000     91252      4 min |    0.07    0.07    0.28    1.12    3.43
     2000    183800      9 min |    0.14    0.14    0.56    2.26    6.92
     5000    459637     22 min |    0.35    0.35    1.41    5.65   17.30
    10000    920809     44 min |    0.71    0.71    2.83   11.31   34.65


## 7. Choose the episode subset

All variants are cached for the **same** episodes (the first `N_EPISODES` of the dataset), so no variant sees
more data than another. `None` means the whole dataset. Colab's disk is wiped at the end of a session; a
free Google Drive holds 15 GB.

In [8]:
N_EPISODES = None if SMOKE else 2000       # <- set from the table above

n_ep = N_EPISODES_TOTAL if N_EPISODES is None else min(N_EPISODES, N_EPISODES_TOTAL)
N_FRAMES = int(lengths[:n_ep].sum())
TAG = f"{DATASET}_ep{n_ep}"
FILES = {name: FEATURES_DIR / f"{TAG}_{name}.h5" for name in VARIANTS}
print(f"{n_ep} episodes, {N_FRAMES} frames -> files {TAG}_<variant>.h5 in {FEATURES_DIR}")

2000 episodes, 183800 frames -> files tworoom_ep2000_<variant>.h5 in /content/stablewm/features


## 8. Extraction

One pass over the frames writes all five variants. A background thread reads the next batch from the
dataset while the GPU encodes the current one. Progress is stored in the files (`frames_done`), so a
rerun resumes.

In [9]:


def open_feature_files():
    files = {}
    for name, path in FILES.items():
        f = h5py.File(path, "a")
        if "feat" not in f:
            k = VARIANTS[name]
            f.create_dataset("feat", shape=(N_FRAMES, k, FEAT_DIM), dtype="float16",
                             chunks=(min(256, N_FRAMES), k, FEAT_DIM))
            with h5py.File(H5_PATH, "r") as src:
                f.create_dataset("action", data=src["action"][:N_FRAMES])
                f.create_dataset("ep_len", data=lengths[:n_ep])
                f.create_dataset("ep_offset", data=offsets[:n_ep])
            f.attrs.update(dict(variant=name, tokens_per_frame=k, feat_dim=FEAT_DIM, source=str(H5_PATH.name),
                                n_episodes=n_ep, n_frames=N_FRAMES, encoder="dinov2_vits14", layer="last",
                                encoder_px=ENCODER_PX, normalization="(x/255 - 0.5) / 0.5", frames_done=0))
        files[name] = f
    return files


def read_batches(start, q):
    with h5py.File(H5_PATH, "r") as src:
        for i in range(start, N_FRAMES, BATCH):
            q.put((i, src["pixels"][i:min(i + BATCH, N_FRAMES)]))
    q.put(None)


files = open_feature_files()
start = min(int(f.attrs["frames_done"]) for f in files.values())
print(f"resuming at frame {start} of {N_FRAMES}" if start else f"extracting {N_FRAMES} frames")

q = queue.Queue(maxsize=3)
threading.Thread(target=read_batches, args=(start, q), daemon=True).start()
t0 = time.perf_counter()
with tqdm(total=N_FRAMES, initial=start, unit="frame") as bar:
    while (item := q.get()) is not None:
        i, frames = item
        feats = encode_and_pool(frames)
        for name, f in files.items():
            f["feat"][i:i + len(frames)] = feats[name]
        if (i // BATCH) % 50 == 0:                       # persist progress now and then
            for f in files.values():
                f.attrs["frames_done"] = i + len(frames)
                f.flush()
        bar.update(len(frames))
for f in files.values():
    f.attrs["frames_done"] = N_FRAMES
    f.close()
print(f"done in {(time.perf_counter() - t0) / 60:.1f} min")

extracting 183800 frames


  0%|          | 0/183800 [00:00<?, ?frame/s]

done in 9.5 min


## 9. Check the files

Shapes, finite values, and a recomputation of a few random frames against what was stored.

In [10]:
rng = np.random.default_rng(0)
check = np.sort(rng.choice(N_FRAMES, size=min(8, N_FRAMES), replace=False))
with h5py.File(H5_PATH, "r") as src:
    fresh = encode_and_pool(src["pixels"][check])
for name, path in FILES.items():
    with h5py.File(path, "r") as f:
        feat = f["feat"]
        stored = feat[check]
        assert f.attrs["frames_done"] == N_FRAMES, f"{name}: incomplete"
        assert np.isfinite(feat[:1000]).all() and np.isfinite(feat[-1000:]).all(), f"{name}: non-finite values"
        diff = np.abs(stored.astype(np.float32) - fresh[name].astype(np.float32)).max()
        print(f"{name:5s} {tuple(feat.shape)} | {path.stat().st_size / 1e9:5.2f} GB | max diff on recompute {diff:.2e}")

cls   (183800, 1, 384) |  0.14 GB | max diff on recompute 9.77e-04
mean  (183800, 1, 384) |  0.14 GB | max diff on recompute 1.95e-03
g2    (183800, 4, 384) |  0.57 GB | max diff on recompute 1.95e-03
g4    (183800, 16, 384) |  2.26 GB | max diff on recompute 3.91e-03
g7    (183800, 49, 384) |  6.92 GB | max diff on recompute 3.91e-03


## 10. Keep the files (Colab)

Colab's disk is wiped at the end of the session. Mount Google Drive and copy the files that fit; the
large ones can be regenerated by rerunning this notebook.

In [11]:
DRIVE_LIMIT_GB = 12   # total to copy; free Drive holds 15 GB

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/dino-wm-tokens/features")
    dest.mkdir(parents=True, exist_ok=True)
    budget = DRIVE_LIMIT_GB * 1e9
    for name, path in sorted(FILES.items(), key=lambda kv: kv[1].stat().st_size):
        size = path.stat().st_size
        if size > budget:
            print(f"skip {path.name} ({size / 1e9:.1f} GB): over the remaining budget")
            continue
        shutil.copy(path, dest / path.name)
        budget -= size
        print(f"copied {path.name} ({size / 1e9:.2f} GB)")
else:
    print("not on Colab: files stay in", FEATURES_DIR)

Mounted at /content/drive
copied tworoom_ep2000_cls.h5 (0.14 GB)
copied tworoom_ep2000_mean.h5 (0.14 GB)
copied tworoom_ep2000_g2.h5 (0.57 GB)
copied tworoom_ep2000_g4.h5 (2.26 GB)
copied tworoom_ep2000_g7.h5 (6.92 GB)
